# Somatic Mutation Preprocessing — TCGA-BRCA
Produces 8 experimental dataset variants in `statistical_filtered/`.

Notebook location: `Thesis_v3/01_Causal_feature_extraction/Mutations/`

Mutations-specific notes vs RNA/CNV:
- Binary data (0/1) — no log2 transform, no z-score normalization
- Frequency-based pre-filter (variance is not meaningful for binary data)
- Extreme sparsity: most genes mutated in <5% of samples
- FDR significance is rarely achieved; fallback to composite ranking is expected

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Mutation frequency > 98.3 percentile |
| 2 | conservative | Mutation frequency > 97.5 percentile |
| 3 | standard | Mutation frequency > 95.9 percentile |
| 4 | fdr_significant | Spearman FDR < 0.05, top 1000 by composite (fallback if none) |
| 5 | balanced | Freq top 25% then Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 percentile |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |


In [2]:
from pathlib import Path

# Notebook: Thesis_v3/01_Causal_feature_extraction/Mutations/mutations_preprocessing.ipynb
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.somaticmutation_wxs.tsv").exists()
)

MUT_FILE     = BASE_DIR / "data" / "TCGA-BRCA.somaticmutation_wxs.tsv"
OUTCOME_FILE = BASE_DIR / "data" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "01_Causal_feature_extraction" / "Mutations" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "01_Causal_feature_extraction" / "Mutations" / "mut_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_GENES    = 50
TARGET_BROAD = 1000

assert MUT_FILE.exists(),     f"Not found: {MUT_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base   : {BASE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print(f"Cache  : {STATS_CACHE}")
print("Paths OK")


Base   : C:\Users\olegk\Desktop\Thesis_v3
Output : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Mutations\statistical_filtered
Cache  : C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Mutations\mut_statistics_cache.csv
Paths OK


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings("ignore")


In [4]:
print("Loading mutation data...")
mut_raw = pd.read_csv(MUT_FILE, sep="\t", index_col=0)
print(f"  Raw shape : {mut_raw.shape}")
print(f"  Columns   : {mut_raw.columns.tolist()[:6]}")

# Build binary matrix: samples × genes (1 = mutated, 0 = not)
# Input is long-format: index=sample_id, 'gene' column=Hugo symbol
mut_matrix = mut_raw.groupby([mut_raw.index, "gene"]).size().unstack(fill_value=0)
mut_binary  = (mut_matrix > 0).astype(int)
print(f"  Binary matrix: {mut_binary.shape}  (samples x genes)")

if mut_binary.columns.duplicated().any():
    mut_binary = mut_binary.loc[:, ~mut_binary.columns.duplicated(keep="first")]
    print(f"  After dedup  : {mut_binary.shape}")

print("Loading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples  : {len(outcome)}")
print(f"  OS.time  : {outcome['OS.time'].min():.0f} - {outcome['OS.time'].max():.0f} days")
print(f"  Events   : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

common  = sorted(set(mut_binary.index) & set(outcome.index))
mut     = mut_binary.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap  : {len(common)} samples")


Loading mutation data...
  Raw shape : (89569, 11)
  Columns   : ['gene', 'chrom', 'start', 'end', 'ref', 'alt']
  Binary matrix: (985, 16662)  (samples x genes)
Loading outcome...
  Samples  : 1076
  OS.time  : 1 - 8605 days
  Events   : 150 (13.9%)
  Overlap  : 948 samples


In [5]:
print("Frequency pre-filter: keep genes mutated in >1% of samples...")

freq_pct = (mut.sum(axis=0) / len(mut) * 100).sort_values(ascending=False)

print(f"  Total genes          : {len(freq_pct):,}")
print(f"  Mutated in >10%      : {(freq_pct > 10).sum():,} genes")
print(f"  Mutated in  >5%      : {(freq_pct >  5).sum():,} genes")
print(f"  Mutated in  >1%      : {(freq_pct >  1).sum():,} genes")
print(f"  Max frequency        : {freq_pct.max():.1f}%  ({freq_pct.index[0]})")
print(f"  Mean frequency       : {freq_pct.mean():.2f}%")
print(f"  Median frequency     : {freq_pct.median():.2f}%")

mut_filt = mut.loc[:, freq_pct > 1.0]
freq_pct = freq_pct[freq_pct > 1.0]   # keep in sync
print(f"  After >1% filter     : {mut_filt.shape[1]:,} genes")
print()
print("Note: no z-score normalization applied — mutation data is binary (0/1).")


Frequency pre-filter: keep genes mutated in >1% of samples...
  Total genes          : 16,662
  Mutated in >10%      : 6 genes
  Mutated in  >5%      : 21 genes
  Mutated in  >1%      : 1,538 genes
  Max frequency        : 34.4%  (TP53)
  Mean frequency       : 0.48%
  Median frequency     : 0.32%
  After >1% filter     : 1,538 genes

Note: no z-score normalization applied — mutation data is binary (0/1).


In [6]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics ({STATS_CACHE.name})...")
    stats = pd.read_csv(STATS_CACHE)
    print(f"  Loaded: {len(stats):,} genes")
else:
    import time
    os_time = outcome["OS.time"].values
    genes   = mut_filt.columns.tolist()
    n       = len(genes)
    print(f"Computing statistics for {n:,} genes...")

    # 1. Spearman correlation
    print("  [1/3] Spearman correlation...")
    t0 = time.time()
    rho_arr, pval_arr = [], []
    for i, g in enumerate(genes):
        r, p = spearmanr(mut_filt[g].values, os_time)
        rho_arr.append(0.0 if np.isnan(r) else float(r))
        pval_arr.append(1.0 if np.isnan(p) else float(p))
        if (i + 1) % 500 == 0:
            print(f"    {i+1:,} / {n:,}  ({time.time()-t0:.0f}s)")
    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")

    # 2. Mutual information
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        mut_filt.values, os_time, random_state=42, n_neighbors=5
    )

    # 3. Distance correlation (rank-based approximation)
    # Full O(n^2) distance correlation is prohibitive even at ~1500 genes
    # given the binary nature of the data (many ties reduce benefit of exact dcor).
    # Rank-based Pearson serves as a consistent proxy across all three modalities.
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for g in genes:
        x_rank = rankdata(mut_filt[g].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    stats = pd.DataFrame({
        "gene":          genes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "mutation_freq": freq_pct[genes].values,
    })

    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0
    stats = stats.sort_values("composite").reset_index(drop=True)

    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE.name}")

n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"  FDR < 0.05  : {n_fdr:,} / {len(stats):,} genes")
if n_fdr == 0:
    print("  Note: 0 FDR-significant genes is expected for mutation data.")
    print("  Individual binary mutation signals are too sparse to reach")
    print("  significance after multiple testing correction. V4 will use")
    print("  composite ranking fallback.")
print(f"  Top 10 by composite score:")
print(stats[["gene","abs_spearman","mutual_info","distance_corr","mutation_freq","composite"]].head(10).to_string(index=False))


Computing statistics for 1,538 genes...
  [1/3] Spearman correlation...
    500 / 1,538  (0s)
    1,000 / 1,538  (1s)
    1,500 / 1,538  (1s)
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to mut_statistics_cache.csv
  FDR < 0.05  : 0 / 1,538 genes
  Note: 0 FDR-significant genes is expected for mutation data.
  Individual binary mutation signals are too sparse to reach
  significance after multiple testing correction. V4 will use
  composite ranking fallback.
  Top 10 by composite score:
    gene  abs_spearman  mutual_info  distance_corr  mutation_freq  composite
   MAML2      0.124536     0.007806       0.124536       1.160338   7.000000
 CACNA1S      0.099078     0.012068       0.099078       1.687764   8.333333
ADAMTS20      0.104914     0.008659       0.104914       1.793249   9.333333
  PTCHD3      0.109671     0.007170       0.109671       1.160338  11.000000
   DOCK9      0.110195     0.006490       0.110195       1.265823  15.000000

In [7]:
print("Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")

def build(gene_list, min_g=MIN_GENES):
    """Select genes; pad with top composite genes if below minimum.
    No normalization — binary data is used as-is."""
    gene_list = [g for g in gene_list if g in mut_filt.columns]
    if len(gene_list) < min_g:
        have = set(gene_list)
        gene_list += [g for g in stats["gene"] if g not in have][:min_g - len(gene_list)]
    return mut_filt[gene_list]

created = []

# V1 Ultra-Conservative: frequency > 98.3 percentile
df    = build(stats[stats["mutation_freq"] > stats["mutation_freq"].quantile(0.983)]["gene"].tolist())
fname = f"mut_1_ultra_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1","name":"ultra_conservative","n":len(df.columns),"logic":"Freq > 98.3 pct","file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} genes")

# V2 Conservative: frequency > 97.5 percentile
df    = build(stats[stats["mutation_freq"] > stats["mutation_freq"].quantile(0.975)]["gene"].tolist())
fname = f"mut_2_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2","name":"conservative","n":len(df.columns),"logic":"Freq > 97.5 pct","file":fname})
print(f"  V2 conservative       : {len(df.columns):,} genes")

# V3 Standard: frequency > 95.9 percentile
df    = build(stats[stats["mutation_freq"] > stats["mutation_freq"].quantile(0.959)]["gene"].tolist())
fname = f"mut_3_standard_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3","name":"standard","n":len(df.columns),"logic":"Freq > 95.9 pct","file":fname})
print(f"  V3 standard           : {len(df.columns):,} genes")

# V4 FDR-significant (expected: 0 genes -> fallback to composite top 1000)
fdr_genes = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["gene"].tolist()
df    = build(fdr_genes, min_g=TARGET_BROAD)
fname = f"mut_4_fdr_significant_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4","name":"fdr_significant","n":len(df.columns),"logic":"FDR<0.05, top 1000 composite fallback","file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} genes  (raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 Balanced: freq top 25% then Spearman top 10%
freq_sub    = stats[stats["mutation_freq"] > stats["mutation_freq"].quantile(0.75)]
corr_thresh = freq_sub["abs_spearman"].quantile(0.90)
df    = build(freq_sub[freq_sub["abs_spearman"] > corr_thresh]["gene"].tolist())
fname = f"mut_5_balanced_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5","name":"balanced","n":len(df.columns),"logic":"Freq top 25% -> Spearman top 10%","file":fname})
print(f"  V5 balanced           : {len(df.columns):,} genes")

# V6 Correlation-based: abs Spearman > 97.5 percentile
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["gene"].tolist())
fname = f"mut_6_correlation_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6","name":"correlation","n":len(df.columns),"logic":"Abs Spearman > 97.5 pct","file":fname})
print(f"  V6 correlation        : {len(df.columns):,} genes")

# V7 Top Correlated: top 100 pos + top 100 neg Spearman
# Using 100+100 to match RNA and CNV pipelines
top_pos = stats[stats["spearman_rho"] > 0].head(100)["gene"].tolist()
top_neg = stats[stats["spearman_rho"] < 0].tail(100)["gene"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
stats[stats["gene"].isin(set(top_pos + top_neg))][
    ["gene","spearman_rho","pval","fdr","mutation_freq"]
].to_csv(OUTPUT_DIR / "mut_7_top_correlated_annotated.csv", index=False)
fname = f"mut_7_top_correlated_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7","name":"top_correlated","n":len(df.columns),"logic":"Top 100 pos + top 100 neg Spearman","file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} genes")

# V8 Composite: top TARGET_BROAD by average rank
# Mutation data is sparse, so TARGET_BROAD may exceed available genes;
# build() will pad with remaining top composite genes up to min_g=MIN_GENES.
df    = build(stats.head(TARGET_BROAD)["gene"].tolist(), min_g=TARGET_BROAD)
fname = f"mut_8_composite_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8","name":"composite","n":len(df.columns),"logic":"Avg rank Spearman+MI+Dcor, top 1000","file":fname})
print(f"  V8 composite          : {len(df.columns):,} genes")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "mut_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V","name","n","logic"]].to_string(index=False))
print(f"Saved to: {OUTPUT_DIR}")


Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\Mutations\statistical_filtered
  V1 ultra_conservative : 50 genes
  V2 conservative       : 50 genes
  V3 standard           : 61 genes
  V4 fdr_significant    : 1,000 genes  (raw FDR<0.05: 0)
  V5 balanced           : 50 genes
  V6 correlation        : 50 genes
  V7 top_correlated     : 200 genes
  V8 composite          : 1,000 genes

 V               name    n                                 logic
V1 ultra_conservative   50                       Freq > 98.3 pct
V2       conservative   50                       Freq > 97.5 pct
V3           standard   61                       Freq > 95.9 pct
V4    fdr_significant 1000 FDR<0.05, top 1000 composite fallback
V5           balanced   50      Freq top 25% -> Spearman top 10%
V6        correlation   50               Abs Spearman > 97.5 pct
V7     top_correlated  200    Top 100 pos + top 100 neg Spearman
V8          composite 1000   Avg rank Spe